In [102]:
import math
import random

class Value:

    def __init__(self, data, children=()):
        self.data = data
        self.children = set(children)
        self.grad = 0.0
        self.backward = lambda : None

    def __add__(self, other):
        if not isinstance(other, Value):
            other = Value(other)
        out = Value(self.data + other.data, (self, other))
        def backward():
            self.grad += out.grad
            other.grad += out.grad

        out.backward = backward
        return out

    def __mul__(self, other):
        if not isinstance(other, Value):
            other = Value(other)
        out = Value(self.data * other.data, (self, other))
        def backward():
            self.grad += out.grad * other.data
            other.grad += out.grad * self.data

        out.backward = backward
        return out

    def __pow__(self, other):
        out = Value(self.data ** other, (self, ))
        def backward():
            self.grad += other * (self.data ** (other - 1)) * out.grad

        out.backward = backward
        return out

    def __truediv__(self, other):
        return self * other**-1

    def __sub__(self, other):
        return self + (other * -1)

    def tanh(self):
        x = self.data
        t = (math.exp(2*x) - 1) / (math.exp(2*x) + 1)
        out = Value(t, (self, ))
        def backward():
            self.grad += (1 - t**2) * out.grad
            
        out.backward = backward
        return out

    def __repr__(self):
        return f'Value: {self.data} Grad: {self.grad}'

    def backwards(self):
        topo = []
        visited = set()
        def topo_sort(node):
            if node not in visited:
                visited.add(node)
                for child in node.children:
                    topo_sort(child)
                topo.append(node)

        self.grad = 1.0
        topo_sort(self)

        for node in reversed(topo):
            node.backward()

class Neuron:

    def __init__(self, nin):
        self.weights = [Value(random.uniform(-1, 1)) for _ in range(nin)]
        self.bias = Value(random.uniform(-1, 1))
        self.parameters = [self.bias] + self.weights

    def __call__(self, x):
        act = sum([wi * xi  for wi, xi in zip(self.weights, x)], self.bias)
        return act.tanh()

class Layer:

    def __init__(self, nin, nout):
        self.neurons = [Neuron(nin) for _ in range(nout)]
        self.parameters = []
        for n in self.neurons:
            for p in n.parameters:
                self.parameters.append(p)

    def __call__(self, x):
        return [n(x) for n in self.neurons]

class MLP:

    def __init__(self, nin, nouts):
        size = [nin] + nouts
        self.layers = [Layer(size[i], size[i + 1]) for i in range(len(nouts))]
        self.parameters = []
        for l in self.layers:
            for p in l.parameters:
                self.parameters.append(p)

    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

model = MLP(784, [16, 16, 10])

print(len(model.parameters))


13002


In [103]:
from torchvision import datasets

raw_train_data = datasets.MNIST(root='../data', download=True, train=True)
raw_test_data = datasets.MNIST(root='../data', download=True, train=False)

train_data = []
test_data = []

for image, label in raw_train_data:
    image_data = [p / 255 for p in list(image.get_flattened_data())]
    train_data.append([image_data, label])

for image, label in raw_test_data:
    image_data = [p / 255 for p in list(image.get_flattened_data())]
    test_data.append([image_data, label])


In [ ]:
def MSELoss(prediction, target):
    return sum([(p - t)**2 for p, t in zip(prediction, target)], Value(0)) / Value(len(prediction))

epochs = 1000
learning_rate = 0.004

for batch, (X, y) in enumerate(train_data):

    for p in model.parameters:
        p.grad = 0.0

    prediction = model(X)
    target = [0.0 for _ in range(10)]
    target[y] = 1.0
    loss = MSELoss(prediction, target)
    loss.backwards()

    if batch % 100 == 0:
        print(loss.data)

    for p in model.parameters:
        p.data += -learning_rate * p.grad
        

0.786691593179322
0.7251716433308547
0.3542561780995472
0.579679702821898
0.5593459302286735
0.7510678684124698
0.4817901202543509
0.4604066604358389
0.5005010320850186
0.5292404275083944
0.44097726826965994
0.5075650463188888
0.48030005561150135
0.5974316287770765
0.6253574692341521
0.7567039091433526
0.5047756610496438
0.42153361282735907
0.5003591508218882
0.5798486826704852
0.6084939026234499
0.40196948272906124
0.3488594800606337
0.5436730256730677
0.4737849705068028
0.4263221597986864
0.6629545954556616
0.6058541482787384
0.2345446407527303
0.4658754098462902
0.5659626614008592
0.22420562628002025
0.48156834734885595
0.3400110895642224
0.4495134976839717
